# TEST Drive FMU

In [11]:
from pathlib import Path
from OMPython import ModelicaSystem

pkg = Path.cwd().parent / 'ControlledPendulum/package.mo'
pkg_name = 'ControlledPendulum'
model_name = 'Drive'
model_str = f'{pkg_name}.{model_name}'

model = ModelicaSystem(str(pkg), model_str, commandLineOptions="--fmiFlags=s:cvode")
model.buildModel()

fmu_path = model.convertMo2Fmu(version='2.0', fmuType='cs')


Notification: Automatically loaded package Complex 4.0.0 due to uses annotation from Modelica.
Notification: Automatically loaded package ModelicaServices 4.0.0 due to uses annotation from Modelica.
Notification: Automatically loaded package Modelica 4.0.0 due to usage.




In [12]:
from fmpy import read_model_description, extract, instantiate_fmu
from fmpy.fmi2 import FMU2Slave

model_description = read_model_description(fmu_path)
unzipdir = extract(fmu_path)
fmu = instantiate_fmu(unzipdir, model_description)

LOG_SOLVER        | info    | CVODE linear multistep method CV_BDF
LOG_SOLVER        | info    | CVODE maximum integration order CV_ITER_NEWTON
LOG_SOLVER        | info    | CVODE use equidistant time grid YES
LOG_SOLVER        | info    | CVODE Using relative error tolerance 1.000000e-06
LOG_SOLVER        | info    | CVODE Using dense internal linear solver SUNLinSol_Dense.
LOG_SOLVER        | info    | CVODE Use internal dense numeric jacobian method.
LOG_SOLVER        | info    | CVODE uses internal root finding method NO
LOG_SOLVER        | info    | CVODE maximum absolut step size 0
LOG_SOLVER        | info    | CVODE initial step size is set automatically
LOG_SOLVER        | info    | CVODE maximum integration order 5
LOG_SOLVER        | info    | CVODE maximum number of nonlinear convergence failures permitted during one step 10
LOG_SOLVER        | info    | CVODE BDF stability limit detection algorithm OFF


In [16]:
fmu.instantiate()
fmu.setupExperiment(startTime=0.0, stopTime=10.0, tolerance=1e-6)
fmu.enterInitializationMode()
fmu.exitInitializationMode()

LOG_SOLVER        | info    | CVODE linear multistep method CV_BDF
LOG_SOLVER        | info    | CVODE maximum integration order CV_ITER_NEWTON
LOG_SOLVER        | info    | CVODE use equidistant time grid YES
LOG_SOLVER        | info    | CVODE Using relative error tolerance 1.000000e-06
LOG_SOLVER        | info    | CVODE Using dense internal linear solver SUNLinSol_Dense.
LOG_SOLVER        | info    | CVODE Use internal dense numeric jacobian method.
LOG_SOLVER        | info    | CVODE uses internal root finding method NO
LOG_SOLVER        | info    | CVODE maximum absolut step size 0
LOG_SOLVER        | info    | CVODE initial step size is set automatically
LOG_SOLVER        | info    | CVODE maximum integration order 5
LOG_SOLVER        | info    | CVODE maximum number of nonlinear convergence failures permitted during one step 10
LOG_SOLVER        | info    | CVODE BDF stability limit detection algorithm OFF


0

In [20]:
md = read_model_description(fmu_path)
vars = {var.name: var for var in md.modelVariables}

fmu.setReal([vars['u_control'].valueReference], [1])
fmu.setReal([vars['omega'].valueReference], [0])

time = 0.0
step_size = 0.01

fmu.doStep(currentCommunicationPoint=time, communicationStepSize=step_size)

In [21]:
torque = fmu.getReal([vars['torque'].valueReference])[0]
print(f'Torque at time {time:.2f} s: {torque:.4f} Nm')

Torque at time 0.00 s: 162.1186 Nm
